# Universal Transformer: a recurrent core

**Universal Transformers** · Dehghani et al. · 2018

[Original paper](https://arxiv.org/abs/1807.03819)

**Learning goal:** Verify sharing, depth signals, and full versus truncated backpropagation.

## Before starting

Attention, residual blocks, and introductory looped transformers.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

Universal Transformers recurrently apply a shared transformation over depth while processing positions in parallel. Position and computation-step signals can be supplied. The paper also studies adaptive per-position halting.

Reusing one nn.Module establishes sharing; separately instantiated modules with equal initial values do not. Computation depth, parameter count, and backward depth are different quantities. We study fixed-depth recurrence and deliberately omit adaptive computation time.

## Reproduction boundary

Fixed-depth shared Transformer only; no adaptive halting, exact original position/depth encoding, or paper tasks.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`h_(s+1) = Core_theta(h_s + depth_signal(s))`, reusing theta at every step.

Full BPTT combines gradient paths through all applications. detach cuts prior history and changes the gradient estimator. Checkpointing can instead trade recomputation for activation memory while preserving mathematical gradients with consistent RNG.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Supply one causal core with deterministic depth signals, adding no trainable parameters per iteration.

In [ ]:
class SharedCore(nn.Module):
    def __init__(self, width=12):
        super().__init__()
        self.core = nn.TransformerEncoderLayer(width, 3, 24, dropout=0, batch_first=True)
        self.width = width
    def forward(self, x, depth, truncate=False):
        h = x
        causal = torch.triu(torch.ones(x.shape[1], x.shape[1], dtype=torch.bool), 1)
        for index in range(depth):
            if truncate and index:
                h = h.detach()
            signal = torch.sin(torch.arange(self.width) / self.width + index) * 0.1
            h = self.core(h + signal, src_mask=causal)
        return h
model = SharedCore(); x = torch.randn(2, 5, 12)
count = sum(p.numel() for p in model.parameters())
for depth in [1, 2, 4]:
    print('depth:', depth, 'parameters:', count, 'output:', model(x, depth).shape)
assert len(list(model.children())) == 1

## Step 2

Keep the forward computation identical and compare only backward behavior.

In [ ]:
target = torch.randn_like(x)
def gradient(truncate):
    model.zero_grad(set_to_none=True)
    output = model(x, 4, truncate)
    F.mse_loss(output, target).backward()
    return output.detach(), model.core.linear1.weight.grad.clone()
full_output, full_gradient = gradient(False)
truncated_output, truncated_gradient = gradient(True)
assert torch.equal(full_output, truncated_output)
print('gradient difference:', (full_gradient - truncated_gradient).norm().item())
assert not torch.allclose(full_gradient, truncated_gradient)

## Step 3

Train at depth 2, then measure several depths. More computation need not improve the result.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
for step in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(x, 2), target); loss.backward(); optimizer.step()
with torch.no_grad():
    depths = [1, 2, 3, 4, 6]
    losses = [F.mse_loss(model(x, d), target).item() for d in depths]
print(dict(zip(depths, losses)))
plt.plot(depths, losses, marker='o', color='#d73535'); plt.xlabel('inference depth'); plt.ylabel('training-batch MSE'); plt.show()

## Reading the result

Loss uses the same tiny training batch: this probes depth sensitivity, not generalization. Identical outputs with different gradients show why detach is not a harmless memory optimization.

## Independent experiments

1. Replace the shared core with four independent cores.
2. Remove depth signals.
3. Randomize training depth from 1 to 4 and compare on held-out data.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

Independent cores increase parameter count. Without depth signals, iteration information comes only through the evolving state. Random depth changes the training distribution; compare equal compute budgets and multiple seeds.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.